# Scaling Laws — Exercises

8 exercises covering scaling law mathematics.

**Exercises:**
1. Power law fitting — fit L(N) and extrapolate
2. Chinchilla optimal allocation — compute N* and D* for given C
3. Under-training ratio — analyse GPT-3
4. IsoFLOP curve — compare configurations at fixed compute
5. Inference-optimal trade-off — 70B vs 7B lifecycle cost
6. Emergent ability metric — continuous vs discrete metrics
7. MoE effective parameters — compute N_eff
8. Scaling law extrapolation with error — propagate uncertainty

Each exercise has: **description → scaffold → solution**

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def fmt_params(n):
    """Format parameter count (e.g., 7.0B, 140M)."""
    if n >= 1e12:
        return f'{n/1e12:.1f}T'
    elif n >= 1e9:
        return f'{n/1e9:.1f}B'
    elif n >= 1e6:
        return f'{n/1e6:.0f}M'
    elif n >= 1e3:
        return f'{n/1e3:.0f}K'
    return str(int(n))

def fmt_sci(x, digits=2):
    """Format number in scientific notation."""
    if x == 0:
        return '0'
    exp = int(np.floor(np.log10(abs(x))))
    mantissa = x / 10**exp
    if abs(exp) <= 2:
        return f'{x:.{digits}f}'
    return f'{mantissa:.{digits}f}×10^{exp}'

# Chinchilla parameters (used across exercises)
CHINCHILLA = {
    'A': 406.4,
    'B': 410.7,
    'alpha': 0.34,
    'beta': 0.28,
    'E': 1.69
}

def chinchilla_loss(N, D, params=CHINCHILLA):
    """L(N, D) = E + A/N^alpha + B/D^beta"""
    p = params
    return p['E'] + p['A'] / N**p['alpha'] + p['B'] / D**p['beta']

print("Setup complete ✓")

---
## Exercise 1: Power Law Fitting

**Given** loss measurements at three model sizes:
- $L(1\text{B}) = 2.8$
- $L(10\text{B}) = 2.4$
- $L(100\text{B}) = 2.1$

**Tasks:**
1. Fit $L(N) = A \cdot N^{-\alpha}$ using log-log linear regression
2. Report fitted $A$ and $\alpha$
3. Estimate $L(1\text{T})$
4. Assess reliability of the extrapolation

In [ ]:
# === Exercise 1: Scaffold ===

# Given data
N_data = np.array([1e9, 10e9, 100e9])    # 1B, 10B, 100B
L_data = np.array([2.8, 2.4, 2.1])       # observed losses

# TODO 1: Take log of both sides to linearise
# log(L) = log(A) - alpha * log(N)
log_N = None  # fill in
log_L = None  # fill in

# TODO 2: Linear regression: y = mx + b
#   where y = log(L), x = log(N), m = -alpha, b = log(A)
alpha_fit = None  # fill in
A_fit = None      # fill in

# TODO 3: Extrapolate to N = 1T
N_target = 1e12
L_predict = None  # fill in

# TODO 4: Print results
# print(f"Fitted: A = {A_fit:.4f}, alpha = {alpha_fit:.6f}")
# print(f"L(1T) = {L_predict:.4f}")
# print(f"Extrapolation factor: {N_target/N_data[-1]:.0f}×")

In [ ]:
# === Exercise 1: Solution ===
np.random.seed(42)

# Given data
N_data = np.array([1e9, 10e9, 100e9])
L_data = np.array([2.8, 2.4, 2.1])

# Step 1: Linearise by taking logarithm
log_N = np.log(N_data)
log_L = np.log(L_data)

# Step 2: Linear regression using least squares
# y = mx + b → log(L) = -alpha * log(N) + log(A)
X = np.column_stack([log_N, np.ones(len(log_N))])
coeffs = np.linalg.lstsq(X, log_L, rcond=None)[0]

alpha_fit = -coeffs[0]   # slope = -alpha
A_fit = np.exp(coeffs[1])  # intercept = log(A)

print("=== Exercise 1: Power Law Fitting ===\n")
print(f"Fitted parameters:")
print(f"  A     = {A_fit:.4f}")
print(f"  alpha = {alpha_fit:.6f}")

# Step 3: Verify fit on training data
print(f"\nVerification:")
for n, l_obs in zip(N_data, L_data):
    l_pred = A_fit * n**(-alpha_fit)
    print(f"  L({fmt_params(n)}): observed = {l_obs:.4f}, predicted = {l_pred:.4f}")

# Step 4: Extrapolate to 1T
N_target = 1e12
L_predict = A_fit * N_target**(-alpha_fit)
print(f"\nExtrapolation:")
print(f"  L({fmt_params(N_target)}) = {L_predict:.4f}")

# Step 5: Assess reliability
extrap_factor = N_target / N_data[-1]
print(f"\nReliability assessment:")
print(f"  Extrapolation factor: {extrap_factor:.0f}× beyond largest training point")
print(f"  Only 3 data points → limited confidence")
print(f"  10× extrapolation is moderate; reasonable if power law holds")
print(f"  Recommendation: train a ~300B model to validate before committing")

---
## Exercise 2: Chinchilla Optimal Allocation

**Given** compute budget $C = 10^{23}$ FLOPs.

**Tasks:**
1. Compute $N^*$ and $D^*$ using the Chinchilla formula
2. Verify $C = 6 N^* D^*$
3. Compute the optimal loss $L^*$
4. Compute $D^*/N^*$ ratio

In [ ]:
# === Exercise 2: Scaffold ===

C = 1e23  # compute budget
p = CHINCHILLA

# Chinchilla optimal allocation formula:
# N* = (A*alpha / (B*beta))^(1/(alpha+beta)) * (C/6)^(beta/(alpha+beta))
# D* = (B*beta / (A*alpha))^(1/(alpha+beta)) * (C/6)^(alpha/(alpha+beta))

# TODO 1: Compute scaling exponents
a = None  # beta / (alpha + beta)  — N exponent
b = None  # alpha / (alpha + beta)  — D exponent

# TODO 2: Compute coefficient
G = None  # (A*alpha / (B*beta))^(1/(alpha+beta))

# TODO 3: Compute N* and D*
N_opt = None
D_opt = None

# TODO 4: Verify C = 6*N*D and compute L*
# C_check = 6 * N_opt * D_opt
# L_opt = chinchilla_loss(N_opt, D_opt)

In [ ]:
# === Exercise 2: Solution ===
np.random.seed(42)

C = 1e23
p = CHINCHILLA

print("=== Exercise 2: Chinchilla Optimal Allocation ===\n")
print(f"Compute budget: C = {fmt_sci(C)} FLOPs\n")

# Step 1: Scaling exponents
a = p['beta'] / (p['alpha'] + p['beta'])   # N exponent
b = p['alpha'] / (p['alpha'] + p['beta'])   # D exponent
print(f"Scaling exponents:")
print(f"  a (N) = β/(α+β) = {p['beta']}/{p['alpha']+p['beta']:.2f} = {a:.4f}")
print(f"  b (D) = α/(α+β) = {p['alpha']}/{p['alpha']+p['beta']:.2f} = {b:.4f}")
print(f"  a + b = {a+b:.4f} (should be 1.0)")

# Step 2: Coefficient G
G = (p['A'] * p['alpha'] / (p['B'] * p['beta']))**(1 / (p['alpha'] + p['beta']))
print(f"\nCoefficient G = (Aα/Bβ)^(1/(α+β)) = {G:.6f}")

# Step 3: Optimal allocation
N_opt = G * (C / 6)**a
D_opt = (1 / G) * (C / 6)**b

print(f"\nOptimal allocation:")
print(f"  N* = {fmt_params(N_opt)} ({N_opt:.3e})")
print(f"  D* = {fmt_params(D_opt)} ({D_opt:.3e})")
print(f"  D*/N* = {D_opt/N_opt:.1f}")

# Step 4: Verification
C_check = 6 * N_opt * D_opt
print(f"\nVerification:")
print(f"  6 × N* × D* = {fmt_sci(C_check)}")
print(f"  C           = {fmt_sci(C)}")
print(f"  Match: {abs(C_check/C - 1) < 1e-6} (relative error: {abs(C_check/C - 1):.2e})")

# Step 5: Optimal loss
L_opt = chinchilla_loss(N_opt, D_opt)
print(f"\nOptimal loss: L* = {L_opt:.4f} (PPL = {np.exp(L_opt):.2f})")

---
## Exercise 3: Under-Training Ratio

**GPT-3**: $N = 175$B parameters, $D = 300$B tokens.

**Tasks:**
1. Compute Chinchilla-optimal $D$ for $N = 175$B
2. Compute the under-training ratio
3. Estimate loss improvement if trained to Chinchilla-optimal
4. How much compute was "wasted"?

In [ ]:
# === Exercise 3: Scaffold ===

N_gpt3 = 175e9     # 175B parameters
D_gpt3 = 300e9     # 300B tokens

# TODO 1: Chinchilla-optimal D for this N
# Rule of thumb: D_opt ≈ 20 * N
D_optimal = None  # fill in

# TODO 2: Under-training ratio
under_training_ratio = None  # D_optimal / D_gpt3

# TODO 3: Compute actual vs optimal loss
# L_actual = chinchilla_loss(N_gpt3, D_gpt3)
# L_optimal = chinchilla_loss(N_gpt3, D_optimal)
# loss_improvement = L_actual - L_optimal

# TODO 4: Compute wasted compute
# C_actual = 6 * N_gpt3 * D_gpt3
# C_chinchilla = compute needed for a Chinchilla-optimal model
#   that reaches the same loss with correct N/D allocation

In [ ]:
# === Exercise 3: Solution ===
np.random.seed(42)

N_gpt3 = 175e9
D_gpt3 = 300e9

print("=== Exercise 3: GPT-3 Under-Training Analysis ===\n")
print(f"GPT-3: N = {fmt_params(N_gpt3)}, D = {fmt_params(D_gpt3)}\n")

# Step 1: Chinchilla-optimal D for this N
D_optimal = 20 * N_gpt3  # 20 tokens per parameter
print(f"Chinchilla-optimal D for N = {fmt_params(N_gpt3)}:")
print(f"  D_opt = 20 × N = 20 × {fmt_params(N_gpt3)} = {fmt_params(D_optimal)}")

# Step 2: Under-training ratio
under_training_ratio = D_optimal / D_gpt3
print(f"\nUnder-training ratio: {D_optimal:.0e} / {D_gpt3:.0e} = {under_training_ratio:.1f}×")
print(f"  GPT-3 was trained on {under_training_ratio:.1f}× FEWER tokens than optimal")

# Step 3: Compute loss
L_actual = chinchilla_loss(N_gpt3, D_gpt3)
L_optimal = chinchilla_loss(N_gpt3, D_optimal)
loss_improvement = L_actual - L_optimal
ppl_actual = np.exp(L_actual)
ppl_optimal = np.exp(L_optimal)

print(f"\nLoss comparison:")
print(f"  L(actual)  = {L_actual:.4f}  (PPL = {ppl_actual:.2f})")
print(f"  L(optimal) = {L_optimal:.4f}  (PPL = {ppl_optimal:.2f})")
print(f"  Improvement = {loss_improvement:.4f} nats ({loss_improvement/L_actual*100:.1f}% relative)")
print(f"  PPL reduction: {ppl_actual - ppl_optimal:.2f}")

# Step 4: Compute waste analysis
C_actual = 6 * N_gpt3 * D_gpt3
C_chinchilla = 6 * N_gpt3 * D_optimal  # if they'd trained to optimal D
print(f"\nCompute analysis:")
print(f"  C(actual GPT-3)     = {fmt_sci(C_actual)} FLOPs")
print(f"  C(Chinchilla for N) = {fmt_sci(C_chinchilla)} FLOPs")
print(f"  Additional compute needed = {fmt_sci(C_chinchilla - C_actual)} FLOPs")
print(f"  Factor: {C_chinchilla/C_actual:.1f}×")

# Alternative: what Chinchilla-optimal model matches GPT-3's compute?
from scipy.optimize import brentq if False else None  # we'll compute manually
# With C = 6*N_gpt3*D_gpt3, Chinchilla-optimal allocation gives:
p = CHINCHILLA
a_exp = p['beta'] / (p['alpha'] + p['beta'])
b_exp = p['alpha'] / (p['alpha'] + p['beta'])
G = (p['A'] * p['alpha'] / (p['B'] * p['beta']))**(1 / (p['alpha'] + p['beta']))

N_chin = G * (C_actual / 6)**a_exp
D_chin = (1 / G) * (C_actual / 6)**b_exp
L_chin = chinchilla_loss(N_chin, D_chin)

print(f"\nAlternative: Chinchilla-optimal with same compute as GPT-3:")
print(f"  N = {fmt_params(N_chin)}, D = {fmt_params(D_chin)}")
print(f"  L = {L_chin:.4f} (PPL = {np.exp(L_chin):.2f})")
print(f"  → Smaller model ({fmt_params(N_chin)} vs {fmt_params(N_gpt3)}) reaches LOWER loss")

---
## Exercise 4: IsoFLOP Curve

At $C = 10^{21}$ FLOPs, compare three configurations:
- $(N = 1\text{B}, D = 167\text{B})$
- $(N = 3\text{B}, D = 56\text{B})$
- $(N = 10\text{B}, D = 17\text{B})$

**Tasks:**
1. Verify each satisfies $C = 6ND$
2. Compute $L$ for each using Chinchilla formula
3. Which is most compute-efficient?
4. Find the analytical optimal $N^*$

In [ ]:
# === Exercise 4: Scaffold ===

C = 1e21  # FLOPs
configs = [
    (1e9,  167e9),   # 1B params, 167B tokens
    (3e9,  56e9),    # 3B params, 56B tokens
    (10e9, 17e9),    # 10B params, 17B tokens
]

# TODO 1: Verify C ≈ 6ND for each
# for N, D in configs:
#     C_check = 6 * N * D
#     ...

# TODO 2: Compute loss for each
# for N, D in configs:
#     L = chinchilla_loss(N, D)
#     ...

# TODO 3: Find best configuration

# TODO 4: Compute analytical N*

In [ ]:
# === Exercise 4: Solution ===
np.random.seed(42)

C = 1e21
configs = [
    (1e9,  167e9),
    (3e9,  56e9),
    (10e9, 17e9),
]

print("=== Exercise 4: IsoFLOP Curve ===\n")
print(f"Target compute: C = {fmt_sci(C)} FLOPs\n")

# Step 1 & 2: Verify and compute loss
p = CHINCHILLA
print(f"{'N':>10} {'D':>12} {'C_check':>14} {'L_param':>10} {'L_data':>10} {'L_total':>10}")
print("-" * 70)

best_L = np.inf
best_config = None

for N, D in configs:
    C_check = 6 * N * D
    L_param = p['A'] / N**p['alpha']
    L_data = p['B'] / D**p['beta']
    L_total = p['E'] + L_param + L_data
    
    print(f"{fmt_params(N):>10} {fmt_params(D):>12} {fmt_sci(C_check):>14} "
          f"{L_param:>10.4f} {L_data:>10.4f} {L_total:>10.4f}")
    
    if L_total < best_L:
        best_L = L_total
        best_config = (N, D)

# Step 3: Best configuration
print(f"\nBest configuration: N = {fmt_params(best_config[0])}, D = {fmt_params(best_config[1])}")
print(f"  L = {best_L:.4f}")

# Step 4: Analytical optimal
a_exp = p['beta'] / (p['alpha'] + p['beta'])
b_exp = p['alpha'] / (p['alpha'] + p['beta'])
G = (p['A'] * p['alpha'] / (p['B'] * p['beta']))**(1 / (p['alpha'] + p['beta']))

N_opt = G * (C / 6)**a_exp
D_opt = (1 / G) * (C / 6)**b_exp
L_opt = chinchilla_loss(N_opt, D_opt)

print(f"\nAnalytical optimum:")
print(f"  N* = {fmt_params(N_opt)}")
print(f"  D* = {fmt_params(D_opt)}")
print(f"  L* = {L_opt:.4f}")
print(f"  D*/N* = {D_opt/N_opt:.1f}")

# Compare
print(f"\nLoss gap of tested configs vs analytical optimum:")
for N, D in configs:
    L = chinchilla_loss(N, D)
    gap = L - L_opt
    print(f"  ({fmt_params(N)}, {fmt_params(D)}): ΔL = +{gap:.4f}")

---
## Exercise 5: Inference-Optimal Trade-Off

**Model A**: 70B params, 1T token training  
**Model B**: 7B params, 10T token training

Both reach similar loss.

**Tasks:**
1. Compute loss for each model
2. If inference cost ∝ N, which is cheaper to serve? By how much?
3. At what query volume does Model B's higher training cost pay off?

In [ ]:
# === Exercise 5: Scaffold ===

# Model configurations
N_A, D_A = 70e9, 1e12     # 70B params, 1T tokens
N_B, D_B = 7e9, 10e12     # 7B params, 10T tokens

# TODO 1: Compute loss for each
# L_A = chinchilla_loss(N_A, D_A)
# L_B = chinchilla_loss(N_B, D_B)

# TODO 2: Inference cost ratio
# Inference FLOPs per query ∝ N
# cost_ratio = N_A / N_B

# TODO 3: Training cost comparison
# C_train_A = 6 * N_A * D_A
# C_train_B = 6 * N_B * D_B

# TODO 4: Find crossover query volume
# Total lifecycle cost: C_train + 2*N*Q
# Cross when: C_train_A + 2*N_A*Q = C_train_B + 2*N_B*Q
# Solve for Q

In [ ]:
# === Exercise 5: Solution ===
np.random.seed(42)

N_A, D_A = 70e9, 1e12
N_B, D_B = 7e9, 10e12

print("=== Exercise 5: Inference-Optimal Trade-Off ===\n")

# Step 1: Loss comparison
L_A = chinchilla_loss(N_A, D_A)
L_B = chinchilla_loss(N_B, D_B)
print(f"Model A (70B, 1T):  L = {L_A:.4f}  (PPL = {np.exp(L_A):.2f})")
print(f"Model B (7B, 10T):  L = {L_B:.4f}  (PPL = {np.exp(L_B):.2f})")
print(f"Loss difference: {abs(L_A - L_B):.4f} nats")

# Step 2: Inference cost
cost_ratio = N_A / N_B
print(f"\nInference cost:")
print(f"  Model A is {cost_ratio:.0f}× more expensive per query than Model B")
print(f"  Model B saves {(1 - 1/cost_ratio)*100:.0f}% on inference per query")

# Step 3: Training cost
C_train_A = 6 * N_A * D_A
C_train_B = 6 * N_B * D_B
print(f"\nTraining cost:")
print(f"  Model A: {fmt_sci(C_train_A)} FLOPs")
print(f"  Model B: {fmt_sci(C_train_B)} FLOPs")
print(f"  Model B costs {C_train_B/C_train_A:.2f}× as much to train")

# Step 4: Crossover
# C_train_A + 2*N_A*Q = C_train_B + 2*N_B*Q
# Q = (C_train_B - C_train_A) / (2*(N_A - N_B))
tokens_per_query = 500  # average output tokens

if C_train_A != C_train_B and N_A != N_B:
    Q_cross = (C_train_B - C_train_A) / (2 * (N_A - N_B))
    if Q_cross > 0:
        queries_cross = Q_cross / tokens_per_query
        print(f"\nCrossover point (Model B total cost < Model A):")
        print(f"  Q_cross = {fmt_sci(Q_cross)} total inference tokens")
        print(f"  ≈ {fmt_sci(queries_cross)} queries (at {tokens_per_query} tokens/query)")
        
        # Daily queries to break even in various timeframes
        for months in [1, 3, 6, 12]:
            days = months * 30
            daily_q = queries_cross / days
            print(f"  Break even in {months:>2} months: {fmt_sci(daily_q)} queries/day")
    else:
        if C_train_B <= C_train_A:
            print(f"\nModel B is cheaper in BOTH training and inference — always better!")
        else:
            print(f"\nModel A is cheaper per query and cheaper to train — always better.")

---
## Exercise 6: Emergent Ability Metric

**Task**: Design a continuous partial-credit metric for 3-digit addition.

**Exact match**: 0% (all wrong) until model gets ALL digits right → 100%  
**Partial credit**: fraction of digits correct → smooth curve

Compare how the two metrics look as a function of model scale.

In [ ]:
# === Exercise 6: Scaffold ===

# Simulate model capability at different scales
# For 3-digit addition: answer has up to 4 digits
# Model has per-digit accuracy that increases with scale

# TODO 1: Define per-digit accuracy as function of scale
# p_digit(N) = sigmoid((log(N) - threshold) / temperature)

# TODO 2: Exact match = product of per-digit accuracies
# p_exact(N) = p_digit(N)^num_digits

# TODO 3: Partial credit = average of per-digit accuracies
# p_partial(N) = p_digit(N)

# TODO 4: Plot both metrics vs scale

In [ ]:
# === Exercise 6: Solution ===
np.random.seed(42)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def per_digit_accuracy(N, threshold=22, temperature=1.5):
    """Per-digit accuracy as function of log-scale model size.
    
    Smooth sigmoid: 0% at small N, ~100% at large N.
    threshold: log(N) where accuracy = 50%
    temperature: controls steepness
    """
    return sigmoid((np.log(N) - threshold) / temperature)

print("=== Exercise 6: Emergent Ability Metrics ===\n")

# Model sizes spanning 10M to 100B
N_range = np.logspace(7, 11, 200)
num_digits = 4  # 3-digit addition → up to 4-digit answer

# Per-digit accuracy (underlying capability)
p_digit = per_digit_accuracy(N_range)

# Exact match: ALL digits must be correct
p_exact = p_digit**num_digits

# Partial credit: average fraction of digits correct
p_partial = p_digit

# Print key values
print(f"{'N':>12} {'Per-Digit':>10} {'Exact Match':>12} {'Partial':>10}")
print("-" * 48)
for N_val in [1e7, 1e8, 1e9, 3e9, 1e10, 3e10, 1e11]:
    pd = per_digit_accuracy(N_val)
    pe = pd**num_digits
    print(f"{fmt_params(N_val):>12} {pd:>10.1%} {pe:>12.1%} {pd:>10.1%}")

print(f"\nKey insight:")
print(f"  Exact match shows 'emergence' at ~{fmt_params(3e9)}–{fmt_params(1e10)}")
print(f"  Partial credit shows SMOOTH improvement throughout")
print(f"  Same underlying capability → different metric → different story")

# Visualise
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left: both metrics
    ax1.plot(N_range, p_exact * 100, 'r-', linewidth=2, label='Exact Match')
    ax1.plot(N_range, p_partial * 100, 'b-', linewidth=2, label='Partial Credit')
    ax1.set_xscale('log')
    ax1.set_xlabel('Parameters N', fontsize=12)
    ax1.set_ylabel('Accuracy (%)', fontsize=12)
    ax1.set_title('3-Digit Addition: Two Metrics', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Right: log-log to show smooth
    # Add small epsilon to avoid log(0)
    p_exact_safe = np.clip(p_exact, 1e-10, 1)
    ax2.plot(N_range, -np.log(1 - p_partial + 1e-10), 'b-', linewidth=2,
             label='Partial Credit (log-odds)')
    ax2.plot(N_range, -np.log(1 - p_exact_safe + 1e-10), 'r-', linewidth=2,
             label='Exact Match (log-odds)')
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_xlabel('Parameters N', fontsize=12)
    ax2.set_ylabel('-log(1 - accuracy)', fontsize=12)
    ax2.set_title('Log-Scale: Smooth vs Sharp', fontsize=14)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('scaling_emergent_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_emergent_metrics.png")

---
## Exercise 7: MoE Effective Parameters

**Dense model**: $N = 7$B  
**MoE model**: $N_{\text{active}} = 7$B, $E = 8$ experts, $\eta = 0.4$

**Tasks:**
1. Compute $N_{\text{eff}}$ for the MoE model
2. What dense model size is the MoE equivalent to?
3. What's the memory cost ratio?
4. Compare loss at same training data

In [ ]:
# === Exercise 7: Scaffold ===

N_dense = 7e9       # Dense baseline
N_active = 7e9      # MoE active params
E = 8               # Number of experts
eta = 0.4           # Expert efficiency exponent

# TODO 1: Compute N_eff = N_active * E^η
N_eff = None  # fill in

# TODO 2: Equivalent dense model
# N_eff is the equivalent dense model size

# TODO 3: Memory cost
# N_total_moe = N_active * E  (approximate)
# memory_ratio = N_total_moe / N_active

# TODO 4: Loss comparison at D = 2T tokens
# L_dense = chinchilla_loss(N_dense, 2e12)
# L_moe = chinchilla_loss(N_eff, 2e12)

In [ ]:
# === Exercise 7: Solution ===
np.random.seed(42)

N_dense = 7e9
N_active = 7e9
E = 8
eta = 0.4
D = 2e12  # training data

print("=== Exercise 7: MoE Effective Parameters ===\n")

# Step 1: N_eff
N_eff = N_active * E**eta
print(f"MoE configuration:")
print(f"  N_active = {fmt_params(N_active)}")
print(f"  E = {E} experts")
print(f"  η = {eta}")
print(f"  N_eff = N_active × E^η = {fmt_params(N_active)} × {E}^{eta}")
print(f"        = {fmt_params(N_active)} × {E**eta:.3f}")
print(f"        = {fmt_params(N_eff)}")

# Step 2: Equivalent dense
print(f"\nEquivalent dense model: {fmt_params(N_eff)}")
print(f"  MoE with {E} experts ≈ {N_eff/N_active:.2f}× effective parameters")

# Step 3: Memory cost
N_total_moe = N_active * E  # approximate (FFN experts only)
mem_ratio = N_total_moe / N_active
print(f"\nMemory analysis:")
print(f"  N_total (MoE) = {fmt_params(N_total_moe)}")
print(f"  Memory ratio: {mem_ratio:.0f}× vs dense baseline")
print(f"  Inference compute ratio: 1× (same active params)")
print(f"  Effective benefit: {N_eff/N_active:.2f}× at {mem_ratio:.0f}× memory cost")

# Step 4: Loss comparison
L_dense = chinchilla_loss(N_dense, D)
L_moe = chinchilla_loss(N_eff, D)  # MoE acts like larger dense model

print(f"\nLoss comparison (trained on {fmt_params(D)} tokens):")
print(f"  Dense ({fmt_params(N_dense)}):          L = {L_dense:.4f}  (PPL = {np.exp(L_dense):.2f})")
print(f"  MoE ({fmt_params(N_active)} active, E={E}): L = {L_moe:.4f}  (PPL = {np.exp(L_moe):.2f})")
print(f"  Improvement: {L_dense - L_moe:.4f} nats ({(L_dense-L_moe)/L_dense*100:.1f}% relative)")

# Bonus: what if we had a dense model with same total params?
L_dense_big = chinchilla_loss(N_total_moe, D)
print(f"\nComparison with same-memory dense model:")
print(f"  Dense ({fmt_params(N_total_moe)}): L = {L_dense_big:.4f}  (PPL = {np.exp(L_dense_big):.2f})")
print(f"  MoE is worse than full dense but {N_total_moe/N_active:.0f}× cheaper to run")

# Sweep over expert counts
print(f"\n--- Scaling with Expert Count ---")
print(f"{'E':>6} {'N_eff':>10} {'Loss':>8} {'PPL':>8} {'Mem':>8}")
print("-" * 44)
for E_val in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    N_e = N_active * E_val**eta
    L_e = chinchilla_loss(N_e, D)
    mem = N_active * E_val
    print(f"{E_val:>6} {fmt_params(N_e):>10} {L_e:>8.4f} {np.exp(L_e):>8.2f} {fmt_params(mem):>8}")

---
## Exercise 8: Scaling Law Extrapolation with Error

**Given**: $\alpha_N = 0.076 \pm 0.01$ and $L(1\text{B}) = 3.0$.

**Tasks:**
1. Compute $L$ at $N = 1$T using central estimate
2. Compute $L$ at $N = 1$T using $\alpha_N = 0.066$ and $\alpha_N = 0.086$
3. Express the range as percentage uncertainty
4. How does uncertainty grow with extrapolation distance?

In [ ]:
# === Exercise 8: Scaffold ===

# Reference point
N_ref = 1e9       # 1B params
L_ref = 3.0       # loss at reference
alpha_mean = 0.076
alpha_std = 0.01

# Target
N_target = 1e12   # 1T params

# Model: L(N) = A * N^(-alpha)
# Calibrate: A = L_ref * N_ref^alpha

# TODO 1: Central estimate
# A_central = L_ref * N_ref**alpha_mean
# L_central = A_central * N_target**(-alpha_mean)

# TODO 2: Lower/upper bounds
# alpha_low = alpha_mean - alpha_std  → higher loss (less scaling)
# alpha_high = alpha_mean + alpha_std → lower loss (more scaling)

# TODO 3: Percentage uncertainty
# range = L_upper - L_lower
# pct_uncertainty = range / L_central * 100

In [ ]:
# === Exercise 8: Solution ===
np.random.seed(42)

N_ref = 1e9
L_ref = 3.0
alpha_mean = 0.076
alpha_std = 0.01
N_target = 1e12

print("=== Exercise 8: Extrapolation with Uncertainty ===\n")
print(f"Reference: L({fmt_params(N_ref)}) = {L_ref}")
print(f"α = {alpha_mean} ± {alpha_std}")
print(f"Target: N = {fmt_params(N_target)}")
print(f"Extrapolation factor: {N_target/N_ref:.0f}×\n")

# Step 1: Central estimate
# L(N) = A * N^(-alpha)
# A = L_ref * N_ref^alpha  (calibrate from reference point)
A_central = L_ref * N_ref**alpha_mean
L_central = A_central * N_target**(-alpha_mean)
print(f"Central estimate (α = {alpha_mean}):")
print(f"  A = {L_ref} × (10^9)^{alpha_mean} = {A_central:.4f}")
print(f"  L({fmt_params(N_target)}) = {A_central:.4f} × ({fmt_params(N_target)})^(-{alpha_mean})")
print(f"  L({fmt_params(N_target)}) = {L_central:.4f}")

# Step 2: Bounds
alpha_low = alpha_mean - alpha_std   # 0.066 → less scaling → higher loss
alpha_high = alpha_mean + alpha_std  # 0.086 → more scaling → lower loss

A_low = L_ref * N_ref**alpha_low
L_upper = A_low * N_target**(-alpha_low)  # higher loss (less scaling)

A_high = L_ref * N_ref**alpha_high
L_lower = A_high * N_target**(-alpha_high)  # lower loss (more scaling)

print(f"\nBounds:")
print(f"  α = {alpha_low} (pessimistic): L = {L_upper:.4f}")
print(f"  α = {alpha_mean} (central):     L = {L_central:.4f}")
print(f"  α = {alpha_high} (optimistic):  L = {L_lower:.4f}")

# Step 3: Percentage uncertainty
range_width = L_upper - L_lower
pct_uncertainty = range_width / L_central * 100
print(f"\nUncertainty:")
print(f"  Range: {L_lower:.4f} – {L_upper:.4f}")
print(f"  Width: {range_width:.4f}")
print(f"  Percentage uncertainty: ±{pct_uncertainty/2:.1f}% ({pct_uncertainty:.1f}% total)")

# Step 4: Uncertainty vs extrapolation distance
print(f"\n--- Uncertainty Growth with Extrapolation ---")
print(f"{'Target N':>12} {'Factor':>10} {'L_central':>10} {'L_low':>10} {'L_high':>10} {'Unc. %':>10}")
print("-" * 66)

for N_t in [1e9, 1e10, 1e11, 1e12, 1e13]:
    A_c = L_ref * N_ref**alpha_mean
    A_l = L_ref * N_ref**alpha_low
    A_h = L_ref * N_ref**alpha_high
    
    L_c = A_c * N_t**(-alpha_mean)
    L_hi = A_l * N_t**(-alpha_low)
    L_lo = A_h * N_t**(-alpha_high)
    
    factor = N_t / N_ref
    unc = (L_hi - L_lo) / L_c * 100
    print(f"{fmt_params(N_t):>12} {factor:>10.0f}× {L_c:>10.4f} {L_lo:>10.4f} {L_hi:>10.4f} {unc:>9.1f}%")

print(f"\nKey insight: uncertainty grows with extrapolation distance!")
print(f"  At 10×: ~{2:.0f}% uncertainty")
print(f"  At 1000×: ~{pct_uncertainty:.0f}% uncertainty")
print(f"  At 10000×: even larger")
print(f"  Always report confidence intervals on scaling law predictions!")

# Visualise
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    N_plot = np.logspace(9, 13, 200)
    L_c_arr = (L_ref * N_ref**alpha_mean) * N_plot**(-alpha_mean)
    L_hi_arr = (L_ref * N_ref**alpha_low) * N_plot**(-alpha_low)
    L_lo_arr = (L_ref * N_ref**alpha_high) * N_plot**(-alpha_high)
    
    ax.fill_between(N_plot, L_lo_arr, L_hi_arr, alpha=0.3, color='steelblue',
                    label=f'α = {alpha_mean} ± {alpha_std}')
    ax.plot(N_plot, L_c_arr, 'b-', linewidth=2, label=f'Central: α = {alpha_mean}')
    ax.plot(N_plot, L_hi_arr, 'b--', linewidth=1, alpha=0.5)
    ax.plot(N_plot, L_lo_arr, 'b--', linewidth=1, alpha=0.5)
    
    # Reference point
    ax.scatter([N_ref], [L_ref], color='red', s=100, zorder=5, label='Reference')
    
    # Target
    ax.scatter([N_target], [L_central], color='green', marker='*', s=200, zorder=5,
              label=f'Prediction: L = {L_central:.3f}')
    ax.axvline(N_target, color='green', linestyle=':', alpha=0.3)
    
    ax.set_xscale('log')
    ax.set_xlabel('Parameters N', fontsize=12)
    ax.set_ylabel('Loss L', fontsize=12)
    ax.set_title('Scaling Law Extrapolation with Uncertainty', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('scaling_extrapolation_exercise.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_extrapolation_exercise.png")

print("\n" + "=" * 60)
print("ALL EXERCISES COMPLETE")
print("=" * 60)